# 02 - File Router (الموجّه التلقائي للمحرك)

هذا النوتبوك يغطي القسم **6.2** من وثيقة التكليف:

> يجب أن يحتوي المشروع على نقطة تشغيل واحدة تفحص حجم الملف وتقرر المحرك المستخدم.

**المنطق:**
```python
SMALL_FILE_THRESHOLD_MB = 200
if file_size_mb <= SMALL_FILE_THRESHOLD_MB:
    engine = "python_batch"
else:
    engine = "pyspark"
```

**إثبات مطلوب:** لازم نجرب الـ Router على:
1. العينة الصغيرة (`orders_small_sample.csv`, ~42 MB) => المتوقع: `python_batch`
2. الملف الضخم الحقيقي (`orders_huge_mixed_quality.csv`, ~12.6 GB) => المتوقع: `pyspark`

هذا يثبت أن **المسارين يعملان فعليًا** عبر نفس نقطة التشغيل، وليسا برنامجين منفصلين بدون Router (وهو أحد حالات الرفض في القسم 13).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / "src"))

from config import settings
from file_router import choose_engine, print_decision

print(f"SMALL_FILE_THRESHOLD_MB (الحد الفاصل الحالي): {settings.SMALL_FILE_THRESHOLD_MB} MB")

SMALL_FILE_THRESHOLD_MB (الحد الفاصل الحالي): 200.0 MB


## تبرير قيمة الحد الفاصل (SMALL_FILE_THRESHOLD_MB = 200)

نضع الحد عند **200 ميجابايت** للأسباب التالية:

- ملف بهذا الحجم يمكن قراءته بأسلوب Streaming (سطر بسطر) عبر Python العادي خلال ثوانٍ معدودة، دون الحاجة لتحميله كاملاً في الذاكرة، ودون فائدة حقيقية من تشغيل Spark (الذي له تكلفة بدء تشغيل SparkSession وتوزيع Partitions).
- الملفات الأكبر من 200MB (وخصوصًا ملفنا الفعلي البالغ ~12.6GB) تستفيد فعليًا من التوازي الذي يوفره Spark عبر تعدد الأنوية والـ partitions، مما يقلل زمن التنفيذ الكلي بشكل ملموس.
- هذا الرقم قابل للتعديل بالكامل من `config/settings.py` أو متغير البيئة `SMALL_FILE_THRESHOLD_MB`، دون الحاجة لتعديل أي كود.

## التجربة 1: العينة الصغيرة (المتوقع: python_batch)

In [2]:
sample_exists = settings.SAMPLE_FILE_PATH.exists()

if sample_exists:
    decision_sample = choose_engine(settings.SAMPLE_FILE_PATH)
    print_decision(decision_sample)
    assert decision_sample["engine"] == "python_batch", "⚠️ كان المتوقع python_batch للعينة الصغيرة!"
    print("\n[تحقق ناجح] الـ Router اختار python_batch كما هو متوقع للملف الصغير.")
else:
    print(f"العينة غير موجودة بعد في: {settings.SAMPLE_FILE_PATH}")
    print("ارجع لنوتبوك 01 وشغّل خلية إنشاء العينة أولاً.")

== قرار الـ Router ==
id_run       : run_20260822T184250_fd38c5eb
file_name    : orders_small_sample.csv
file_size_mb : 41.765
threshold_mb : 200.0
engine       : python_batch
reason       : حجم الملف (41.77 MB) أقل من أو يساوي الحد الفاصل (200.0 MB) => Batch Python كافٍ وأسرع بدون تكلفة تشغيل Spark.

[تحقق ناجح] الـ Router اختار python_batch كما هو متوقع للملف الصغير.


## التجربة 2: الملف الضخم الحقيقي (المتوقع: pyspark)

In [3]:
source_exists = settings.SOURCE_FILE_PATH.exists()

if source_exists:
    decision_huge = choose_engine(settings.SOURCE_FILE_PATH)
    print_decision(decision_huge)
    assert decision_huge["engine"] == "pyspark", "⚠️ كان المتوقع pyspark للملف الضخم!"
    print("\n[تحقق ناجح] الـ Router اختار pyspark كما هو متوقع للملف الضخم.")
else:
    print(f"الملف الضخم غير موجود في: {settings.SOURCE_FILE_PATH}")

== قرار الـ Router ==
id_run       : run_20260822T184525_9cf560f0
file_name    : orders_huge_mixed_quality.csv
file_size_mb : 12650.32
threshold_mb : 200.0
engine       : pyspark
reason       : حجم الملف (12650.32 MB) أكبر من الحد الفاصل (200.0 MB) => نحتاج معالجة متوازية عبر PySpark.

[تحقق ناجح] الـ Router اختار pyspark كما هو متوقع للملف الضخم.


## ملخص النتائج

| الملف | الحجم | المحرك المختار |
|---|---|---|
| العينة الصغيرة | ~42 MB | python_batch |
| الملف الضخم | ~12.6 GB | pyspark |

كلا المسارين يمران عبر **نفس نقطة الدخول** (`choose_engine`)، ولا يوجد برنامجان منفصلان بدون Router — هذا يحقق شرط القسم 6.2 بالكامل.

## الخطوة القادمة

**`03_python_batch_loader.ipynb`** — بناء محمّل الملفات الصغيرة (Streaming + دفعات + insert_many)، وتشغيله فعليًا على العينة الصغيرة.